In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import os
from datetime import datetime
import threading
import queue  # <--- Added for thread safety

# -----------------------------------------------------------------------------
# BUSINESS LOGIC LAYER (Unchanged)
# -----------------------------------------------------------------------------

class DataProcessor:
    """
    Handles all data ingestion, transformation, and export logic.
    """
    def _clean_part_number(self, val):
        try:
            s_val = str(int(float(val)))
            return s_val.zfill(4)
        except (ValueError, TypeError):
            return str(val).zfill(4)

    def _load_dynamic_header_sheet(self, filepath, sheet_idx):
        try:
            df_raw = pd.read_excel(filepath, sheet_name=sheet_idx, header=None)
            header_row_idx = None
            for idx, row in df_raw.head(20).iterrows():
                if row.astype(str).str.contains("ACCOUNT NO.", case=False, na=False).any():
                    header_row_idx = idx
                    break
            
            if header_row_idx is None:
                raise ValueError(f"Could not find 'ACCOUNT NO.' header in sheet index {sheet_idx}")

            df_raw.columns = df_raw.iloc[header_row_idx]
            df = df_raw.iloc[header_row_idx + 1:].copy()
            df.reset_index(drop=True, inplace=True)

            col_map = {c: 'ACCOUNT NO.' for c in df.columns if 'ACCOUNT' in str(c).upper() and 'NO' in str(c).upper()}
            df.rename(columns=col_map, inplace=True)

            null_mask = df['ACCOUNT NO.'].isna() | (df['ACCOUNT NO.'].astype(str).str.strip() == '')
            if null_mask.any():
                first_blank_idx = null_mask.idxmax()
                if null_mask[first_blank_idx]: 
                    df = df.loc[:first_blank_idx-1]

            return df
        except Exception as e:
            raise RuntimeError(f"Error loading sheet {sheet_idx}: {str(e)}")

    def process_files(self, file_0_path, file_1_path, file_2_path, output_folder):
        generated_files = []
        date_str = datetime.now().strftime("%Y%m%d")

        # LOAD REFERENCE FILES
        try:
            df_parts_master = pd.read_excel(file_1_path)
            df_parts_master['Part No.'] = df_parts_master['Part No.'].apply(self._clean_part_number)
        except Exception as e:
            raise RuntimeError(f"Failed to load Master Part Numbers: {e}")

        try:
            df_po_master = pd.read_excel(file_2_path)
            df_po_master.rename(columns={'Account #': 'ACCOUNT NO.', 'Part #': 'Part No.'}, inplace=True)
            df_po_master['Part No.'] = df_po_master['Part No.'].apply(self._clean_part_number)
        except Exception as e:
            raise RuntimeError(f"Failed to load Master PO Account File: {e}")

        # OUTPUT 1 LOGIC
        try:
            df_eq_tab0 = self._load_dynamic_header_sheet(file_0_path, sheet_idx=0)
            cols_needed = ['ACCOUNT NO.', 'EQUIP', 'QTY']
            df_eq_tab0 = df_eq_tab0[cols_needed].copy()

            dupe_mask = df_eq_tab0.duplicated(subset=['ACCOUNT NO.'], keep=False)
            df_unique = df_eq_tab0[~dupe_mask].copy()
            df_duplicate = df_eq_tab0[dupe_mask].copy()

            df_unique = pd.merge(df_unique, df_parts_master[['Equipment', 'Part No.']], 
                                 left_on='EQUIP', right_on='Equipment', how='left')
            df_duplicate = pd.merge(df_duplicate, df_parts_master[['Equipment', 'Part No.']], 
                                    left_on='EQUIP', right_on='Equipment', how='left')

            df_unique['Part No.'] = df_unique['Part No.'].fillna('0000').apply(self._clean_part_number)
            df_duplicate['Part No.'] = df_duplicate['Part No.'].fillna('0000').apply(self._clean_part_number)

            df_duplicate = df_duplicate[df_duplicate['EQUIP'] != "H/C Filtration -Countertop Unit"].copy()

            dupe_check_mask = df_duplicate.duplicated(subset=['ACCOUNT NO.'], keep=False)
            newly_unique = df_duplicate[~dupe_check_mask].copy()
            still_duplicate = df_duplicate[dupe_check_mask].copy()

            df_output_1 = pd.concat([df_unique, newly_unique], ignore_index=True)
            out1_cols = ['ACCOUNT NO.', 'EQUIP', 'QTY', 'Part No.']
            df_output_1 = df_output_1[out1_cols]

            out1_name = f"UCSD EQUIPMENT Equipment Import {date_str}.xlsx"
            self._save_to_excel(df_output_1, os.path.join(output_folder, out1_name))
            generated_files.append(out1_name)
            
            df_duplicates_final = still_duplicate
        except Exception as e:
            raise RuntimeError(f"Error processing Output 1 Logic: {e}")

        # OUTPUT 2 LOGIC
        try:
            df_eq_tab2 = self._load_dynamic_header_sheet(file_0_path, sheet_idx=2)
            df_eq_tab2 = df_eq_tab2[['ACCOUNT NO.', 'EQUIP', 'QTY']].copy()
            df_eq_tab2 = pd.merge(df_eq_tab2, df_parts_master[['Equipment', 'Part No.']], 
                                  left_on='EQUIP', right_on='Equipment', how='left')
            
            df_output_2 = df_eq_tab2.groupby(['ACCOUNT NO.', 'EQUIP', 'Part No.'], as_index=False)['QTY'].sum()
            df_output_2['Part No.'] = df_output_2['Part No.'].fillna('0000').apply(self._clean_part_number)
            df_output_2 = df_output_2[['ACCOUNT NO.', 'EQUIP', 'QTY', 'Part No.']]

            out2_name = f"UCSD EQUIPMENT STD Import {date_str}.xlsx"
            self._save_to_excel(df_output_2, os.path.join(output_folder, out2_name))
            generated_files.append(out2_name)
        except Exception as e:
            raise RuntimeError(f"Error processing Output 2 Logic: {e}")

        # OUTPUT 3 & 4 LOGIC
        try:
            df_dup_proc = df_duplicates_final[['ACCOUNT NO.', 'EQUIP', 'QTY', 'Part No.']].copy()
            df_dup_grouped = df_dup_proc.groupby(['ACCOUNT NO.', 'Part No.'], as_index=False)['QTY'].sum()

            df_merged = pd.merge(df_dup_grouped, df_po_master, 
                                 on=['ACCOUNT NO.', 'Part No.'], 
                                 how='left', suffixes=('_calc', '_master'))

            def check_match(row):
                q_calc = float(row['QTY']) if pd.notna(row['QTY']) else 0
                q_mast = float(row['Quantity']) if pd.notna(row['Quantity']) else 0
                return "Pass" if q_calc == q_mast else "Mismatch"

            df_merged['Status'] = df_merged.apply(check_match, axis=1)

            df_out3 = df_merged[df_merged['Status'] == 'Pass'].copy()
            cols_final = ['PO #', 'ACCOUNT NO.', 'Equipment', 'Quantity', 'Part No.']
            df_out3 = df_out3[cols_final]
            df_out3.rename(columns={'ACCOUNT NO.': 'Account #', 'Part No.': 'Part #'}, inplace=True)
            
            out3_name = f"UCSD EQUIPMENT Duplicate Import {date_str}.xlsx"
            self._save_to_excel(df_out3, os.path.join(output_folder, out3_name))
            generated_files.append(out3_name)

            df_out4 = df_merged[df_merged['Status'] == 'Mismatch'].copy()
            out4_name = f"UCSD EQUIPMENT Duplicate ERRORS {date_str}.xlsx"
            self._save_to_excel(df_out4, os.path.join(output_folder, out4_name))
            generated_files.append(out4_name)
        except Exception as e:
            raise RuntimeError(f"Error processing Output 3/4 Logic: {e}")

        return generated_files

    def _save_to_excel(self, df, path):
        df.to_excel(path, index=False)


# -----------------------------------------------------------------------------
# GUI LAYER (Refactored for Thread Safety)
# -----------------------------------------------------------------------------

class ModernApp(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("Equipment Processing Tool")
        self.geometry("700x500")
        self.configure(bg="#f0f2f5")
        
        self.processor = DataProcessor()
        self.msg_queue = queue.Queue()  # <--- Thread-safe Queue

        self.file_0_path = tk.StringVar()
        self.file_1_path = tk.StringVar()
        self.file_2_path = tk.StringVar()
        self.output_folder = tk.StringVar()

        self._setup_styles()
        self._create_widgets()

    def _setup_styles(self):
        style = ttk.Style(self)
        style.theme_use('clam')
        style.configure("Card.TFrame", background="#ffffff", relief="flat")
        style.configure("Main.TFrame", background="#f0f2f5")
        style.configure("Header.TLabel", background="#ffffff", foreground="#333333", font=("Segoe UI", 16, "bold"))
        style.configure("Sub.TLabel", background="#ffffff", foreground="#666666", font=("Segoe UI", 10))
        style.configure("Status.TLabel", background="#ffffff", font=("Segoe UI", 12))
        style.configure("Accent.TButton", background="#007acc", foreground="white", font=("Segoe UI", 10, "bold"), borderwidth=0, padding=10)
        style.map("Accent.TButton", background=[('active', '#005fa3')])
        style.configure("Browse.TButton", font=("Segoe UI", 9))

    def _create_widgets(self):
        main_container = ttk.Frame(self, style="Main.TFrame")
        main_container.pack(fill="both", expand=True, padx=20, pady=20)

        card = ttk.Frame(main_container, style="Card.TFrame", padding=30)
        card.pack(fill="both", expand=True)

        ttk.Label(card, text="Equipment Data Processor", style="Header.TLabel").pack(anchor="w", pady=(0, 5))
        ttk.Label(card, text="Select input files and output directory to begin.", style="Sub.TLabel").pack(anchor="w", pady=(0, 20))

        self.f0_status = self._create_file_row(card, "00 Equipment File:", self.file_0_path)
        self.f1_status = self._create_file_row(card, "01 Master File Part Numbers:", self.file_1_path)
        self.f2_status = self._create_file_row(card, "02 Master File PO Accounts:", self.file_2_path)
        
        ttk.Separator(card, orient='horizontal').pack(fill='x', pady=15)
        
        self.out_status = self._create_dir_row(card, "Output Folder:", self.output_folder)

        self.run_btn = ttk.Button(card, text="Run Process", command=self.run_process, style="Accent.TButton", state="disabled")
        self.run_btn.pack(pady=20, fill="x")

        self.status_var = tk.StringVar(value="Ready")
        lbl_log = ttk.Label(card, textvariable=self.status_var, style="Sub.TLabel", wraplength=600)
        lbl_log.pack(side="bottom", anchor="w")

    def _create_file_row(self, parent, label_text, variable):
        frame = ttk.Frame(parent, style="Card.TFrame")
        frame.pack(fill="x", pady=5)
        status_lbl = ttk.Label(frame, text="○", width=3, style="Status.TLabel", foreground="#ccc")
        status_lbl.pack(side="left")
        ttk.Label(frame, text=label_text, width=30, style="Sub.TLabel").pack(side="left")
        entry = ttk.Entry(frame, textvariable=variable, state="readonly", width=40)
        entry.pack(side="left", fill="x", expand=True, padx=5)
        btn = ttk.Button(frame, text="Browse", style="Browse.TButton", command=lambda: self.select_file(variable, status_lbl))
        btn.pack(side="left")
        return status_lbl

    def _create_dir_row(self, parent, label_text, variable):
        frame = ttk.Frame(parent, style="Card.TFrame")
        frame.pack(fill="x", pady=5)
        status_lbl = ttk.Label(frame, text="○", width=3, style="Status.TLabel", foreground="#ccc")
        status_lbl.pack(side="left")
        ttk.Label(frame, text=label_text, width=30, style="Sub.TLabel").pack(side="left")
        entry = ttk.Entry(frame, textvariable=variable, state="readonly", width=40)
        entry.pack(side="left", fill="x", expand=True, padx=5)
        btn = ttk.Button(frame, text="Select", style="Browse.TButton", command=lambda: self.select_dir(variable, status_lbl))
        btn.pack(side="left")
        return status_lbl

    def select_file(self, variable, status_lbl):
        path = filedialog.askopenfilename(filetypes=[("Excel Files", "*.xlsx;*.xls;*.csv")])
        if path:
            variable.set(path)
            status_lbl.config(text="✓", foreground="green")
            self.check_readiness()

    def select_dir(self, variable, status_lbl):
        path = filedialog.askdirectory()
        if path:
            variable.set(path)
            status_lbl.config(text="✓", foreground="green")
            self.check_readiness()

    def check_readiness(self):
        if all([self.file_0_path.get(), self.file_1_path.get(), self.file_2_path.get(), self.output_folder.get()]):
            self.run_btn.config(state="normal")
        else:
            self.run_btn.config(state="disabled")

    # --- THREAD SAFE EXECUTION START ---
    def run_process(self):
        self.run_btn.config(state="disabled")
        self.status_var.set("Processing... Please wait.")
        
        # Start the worker thread
        threading.Thread(target=self._worker_thread, daemon=True).start()
        
        # Start checking the queue
        self.after(100, self._check_queue)

    def _worker_thread(self):
        """Runs in background thread. No GUI calls allowed here."""
        try:
            files = self.processor.process_files(
                self.file_0_path.get(),
                self.file_1_path.get(),
                self.file_2_path.get(),
                self.output_folder.get()
            )
            # Put result in queue
            self.msg_queue.put(("success", files))
        except Exception as e:
            # Put error in queue
            self.msg_queue.put(("error", str(e)))

    def _check_queue(self):
        """Runs in main thread. Checks queue for messages from worker."""
        try:
            msg_type, data = self.msg_queue.get_nowait()
            
            if msg_type == "success":
                self._on_success(len(data))
            elif msg_type == "error":
                self._on_error(data)
                
        except queue.Empty:
            # If queue is empty, check again in 100ms
            self.after(100, self._check_queue)

    def _on_success(self, count):
        self.status_var.set(f"Success! Generated {count} files.")
        messagebox.showinfo("Complete", f"Successfully generated {count} files in the output directory.")
        self.run_btn.config(state="normal")

    def _on_error(self, error_msg):
        self.status_var.set(f"Error: {error_msg}")
        messagebox.showerror("Processing Error", error_msg)
        self.run_btn.config(state="normal")
    # --- THREAD SAFE EXECUTION END ---

if __name__ == "__main__":
    app = ModernApp()
    app.mainloop()

c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
